# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [30]:
# imports
import os
from dotenv import load_dotenv
#from scraper import fetch_website_contents
from IPython.display import Markdown, display, update_display
from openai import OpenAI

## 1) Imports
This section loads the Python libraries needed for the notebook:
- `os` lets us read environment variables.
- `load_dotenv` loads values from a `.env` file.
- `Markdown` and `display` help show formatted output inside Jupyter.
- `OpenAI` creates the client used to call the OpenAI API.


In [1]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

## 2) Model constants
These variables define which AI models the notebook will use.
- `MODEL_GPT` chooses the OpenAI model for the main answer.
- `MODEL_LLAMA` is reserved for a local Ollama model if you want to compare results.
This makes it easy to change models in one place instead of editing multiple requests.


In [3]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


## 3) Environment setup and API key validation
This block reads the OpenAI API key from the environment and checks whether it is valid.
If the key is missing, malformed, or contains extra whitespace, the script prints a helpful troubleshooting message.
This prevents confusing errors later when the API request is sent.


In [ ]:
# Initialize and constants
openai = OpenAI()

## 4) OpenAI client initialization
This creates the client object that will send requests to the OpenAI API.
The notebook uses this client for every question it asks the model.


In [40]:
# Get gpt-4o-mini to answer, with streaming
system_prompt = """
 You're an helpful assistant, who provides the answers/solutions to the given question.
 Provide the response to the question with a clear and concise explanation with a maximum of 200 words.
 Apart from that, provide an example for the response in a frontend development context to understand
 Provide the response in both endglish and Telugu language version one after the other
"""

## 5) System prompt
The system prompt tells the model how to behave.
It sets the role, response length, and style expectations.
This is important because it guides the model toward concise, helpful explanations rather than rambling or off-topic answers.


In [ ]:
def get_code_user_prompt(code):
    user_prompt = f"""
Here is the code -
Please explain this code and why it works the way it does:

"""
    user_prompt += "\n".join(code)
    return user_prompt


## 6) Build the user prompt
This helper function turns the code snippet into a user message for the model.
It adds context like “explain this code and why it works,” then joins the provided lines together.
This makes the API request readable and consistent.


In [37]:
def stream_response_to_display(stream, display_handle):
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)
    return response


## 7) Stream output into the notebook
This helper receives the streaming response from the model and updates the displayed text as chunks arrive.
It accumulates the answer in `response` and refreshes the Jupyter output continuously, which makes the response feel live.


In [ ]:
def get_code_response(code):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_code_user_prompt(code)}
        ],
        stream=True
    )
    display_information = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_information.display_id)
    


## 8) Main API request
This is the main function that sends the prompt to the model.
It creates a streaming chat completion, passes the system message and user message, and then shows the returned answer in the notebook as it arrives.
This pattern is useful for long responses because it reduces waiting time and creates a more interactive experience.


In [ ]:
code1 = "explain stream_response_to_display function"
get_code_response(code1)


## 9) Run the example
This final cell demonstrates how to use the helper function.
It stores a question string and calls the API function, so the notebook produces a streamed answer for that prompt.
You can replace the text with any code snippet or technical question you want to analyze.
